# 05 - Assistants API with Vector Store Intelligence

This notebook demonstrates the most advanced context provision method: the Assistants API with Vector Store. This approach allows AI to have **persistent memory** and **document-based intelligence**.

## 🧠 What Makes Assistants Special
- **Persistent Context**: Conversations and knowledge persist across sessions
- **Document Intelligence**: Upload files for AI to reference and search
- **Vector Store**: Automatic embedding and retrieval of relevant information
- **Stateful Conversations**: No need to manage conversation history manually

## 🎯 Learning Objectives
- Set up an Assistant with Vector Store capabilities
- Upload documents for the AI to reference
- Build document-aware conversational AI
- Understand when Assistants are better than function calling

## 📊 Context Methods Comparison

| Method | Context Source | Persistence | Real-time Data | Best For |
|--------|---------------|-------------|----------------|----------|
| System Prompts | Static text | Per conversation | ❌ No | Domain expertise |
| Multi-turn | Conversation | Per conversation | ❌ No | Dynamic learning |
| Function Calling | External APIs | None | ✅ Yes | Live data access |
| **Assistants + Vector** | **Documents** | **✅ Permanent** | **✅ Searchable** | **Knowledge base AI** |

In [1]:
import os
import time
import json
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Azure OpenAI Configuration for Assistants
# Following Microsoft Learn documentation: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-05-01-preview",  # Microsoft recommends 2024-02-15-preview minimum, latest is better
)

deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")

print("✅ Azure OpenAI client initialized for Assistants API")
print(f"🤖 Model deployment: {deployment}")
print(f"📅 API version: 2024-05-01-preview")
print(f"📄 Following Microsoft Learn guidance: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant")
print("✅ Vector Stores are now out of beta - using modern API!")

✅ Azure OpenAI client initialized for Assistants API
🤖 Model deployment: gpt-4o
📅 API version: 2024-05-01-preview
📄 Following Microsoft Learn guidance: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant
✅ Vector Stores are now out of beta - using modern API!


In [2]:
# 🔍 System Diagnostics
import sys
import openai

print("🐍 Python Version:", sys.version)
print("📦 OpenAI Library Version:", openai.__version__)
print("🌐 API Base URL:", client.base_url)
print("🔑 API Version:", client._api_version)
print("✅ Vector Stores API: Available (non-beta)")

# Test basic APIs
try:
    files_list = client.files.list(limit=1)
    print("✅ Files API: Available")
except Exception as e:
    print(f"❌ Files API: Error - {e}")

print("\n" + "="*50)

🐍 Python Version: 3.13.9 (tags/v3.13.9:8183fa5, Oct 14 2025, 14:09:13) [MSC v.1944 64 bit (AMD64)]
📦 OpenAI Library Version: 1.107.1
🌐 API Base URL: https://foundrynative-resource.openai.azure.com/openai/
🔑 API Version: 2024-05-01-preview
✅ Vector Stores API: Available (non-beta)
✅ Files API: Available



## 🔧 Azure OpenAI Assistants with Vector Store

**This notebook follows the official Microsoft Learn documentation:**
📖 **[Getting started with Azure OpenAI Assistants](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant)**

### 🎉 **Vector Stores are now out of beta!**
- Use `client.vector_stores` (modern API)
- No more beta fallbacks needed
- Production ready implementation

### 📋 **Requirements:**
- ✅ **API Version**: `2024-02-15-preview` or newer
- ✅ **Supported Models**: GPT-4, GPT-4 Turbo, GPT-3.5 Turbo
- ✅ **Regional Support**: Check [Azure OpenAI models page](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/models#assistants-preview)

### 🛠️ **What You'll Learn:**
✅ **Vector Store Creation**: Using `client.vector_stores.create()`  
✅ **File Upload**: Files uploaded with `purpose="assistants"`  
✅ **Assistant Integration**: Using `tool_resources.file_search.vector_store_ids`  
✅ **Document Search**: Semantic search with citations  

---

## 🗃️ Step 1: Create a Vector Store for Documents

**Vector stores automatically parse, chunk, embed, and store files for semantic search.**

In [3]:
# 🗃️ Step 1: Create a Vector Store for Documents
# Following Microsoft Learn guide: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/file-search

def create_vector_store():
    """Create a Vector Store for document storage and retrieval"""
    
    print("📁 Creating Vector Store...")
    
    # Create vector store using modern API (Vector Stores are now out of beta!)
    vector_store = client.vector_stores.create(
        name="MCP Workshop Knowledge Base",
        expires_after={
            "anchor": "last_active_at",
            "days": 7  # Clean up after 7 days of inactivity
        },
        metadata={
            "purpose": "Store workshop documentation and Azure guides",
            "created_by": "mcp-workshop",
            "version": "1.0"
        }
    )
    
    print(f"✅ Vector Store created successfully!")
    print(f"   🆔 ID: {vector_store.id}")
    print(f"   📝 Name: {vector_store.name}")
    print(f"   📊 Status: {vector_store.status}")
    print(f"   ⏰ Expires: 7 days after last activity (cost management)")
    
    return vector_store

# Create the vector store
vector_store = create_vector_store()

📁 Creating Vector Store...
✅ Vector Store created successfully!
   🆔 ID: vs_5mfUwVqVm8jsT7uDv9W1OIFB
   📝 Name: MCP Workshop Knowledge Base
   📊 Status: completed
   ⏰ Expires: 7 days after last activity (cost management)


In [4]:
# Create a sample document for the Vector Store
def create_sample_document():
    """Create a sample Azure documentation file"""
    
    azure_guide_content = """
# Azure OpenAI Service Best Practices Guide

## Overview
Azure OpenAI Service provides enterprise-grade access to OpenAI models like GPT-4, GPT-3.5, and DALL-E with Azure's security and compliance.

## Key Features
- Enterprise security and compliance (SOC 2, GDPR, HIPAA)
- Private networking with Virtual Networks
- Managed identity and RBAC integration
- Built-in content filtering and monitoring
- Multiple model deployments in same resource

## Function Calling Best Practices

### 1. Tool Design Principles
- Keep functions focused and single-purpose
- Use descriptive names and clear descriptions
- Include comprehensive parameter documentation
- Handle errors gracefully with try-catch blocks

### 2. Performance Optimization
- Use parallel tool calls when possible
- Minimize API round-trips
- Cache function results when appropriate
- Monitor token usage and costs

### 3. Security Considerations
- Validate all function parameters
- Implement rate limiting for external API calls
- Use environment variables for sensitive data
- Log function calls for audit purposes

## Cost Management
- Monitor token usage across deployments
- Use content filtering to reduce unwanted calls
- Implement caching strategies
- Consider model selection based on use case

## Regional Availability
- GPT-4o: Available in East US, West Europe, South Central US
- GPT-3.5-turbo: Available in most regions
- DALL-E 3: Limited regional availability
- Check Azure portal for latest regional support

## Monitoring and Logging
- Enable diagnostic settings
- Use Application Insights for detailed telemetry
- Set up alerts for quota limits
- Monitor for content filtering triggers

## Common Issues and Solutions

### High Latency
- Check regional proximity to deployment
- Review concurrent request limits
- Consider connection pooling
- Optimize prompt length

### Rate Limiting
- Implement exponential backoff
- Use multiple deployments for load distribution
- Monitor requests per minute (RPM) limits
- Consider provisioned throughput for predictable workloads

### Content Filtering
- Review and adjust content filtering policies
- Test prompts that might trigger filters
- Implement fallback responses
- Monitor filtering logs

## Integration Patterns

### Direct API Usage
- REST API calls for simple integrations
- SDK usage for complex applications
- Streaming for real-time responses

### Function Calling Workflows
1. Define functions with clear schemas
2. Handle multiple tool calls efficiently
3. Implement proper error handling
4. Provide meaningful function responses

### Assistants API
- Use for stateful conversations
- Leverage Vector Stores for document search
- Implement file upload capabilities
- Monitor assistant performance

## Troubleshooting Checklist
- ✅ API key and endpoint configured correctly
- ✅ Deployment name matches configuration
- ✅ API version supports required features
- ✅ Regional quotas not exceeded
- ✅ Network connectivity to Azure
- ✅ Content filtering policies appropriate
"""
    
    # Save to a file
    doc_path = "azure_openai_guide.txt"
    with open(doc_path, "w", encoding="utf-8") as f:
        f.write(azure_guide_content)
    
    print(f"📄 Sample document created: {doc_path}")
    print(f"   📊 Size: {len(azure_guide_content)} characters")
    
    return doc_path

# Create the sample document
document_path = create_sample_document()

📄 Sample document created: azure_openai_guide.txt
   📊 Size: 3007 characters


## 📤 Step 2: Upload Documents to Vector Store

Now let's upload our Azure guide to the Vector Store so the Assistant can search through it.

In [5]:
# 📤 Step 2: Upload Documents to Vector Store
# Following Microsoft Learn pattern: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/file-search#upload-files-for-file-search

def upload_file_to_vector_store(file_path, vector_store_id):
    """Upload a file to Vector Store using modern API"""
    
    print(f"📤 Uploading {file_path}...")
    
    # Step 1: Upload file to Azure OpenAI (required for all approaches)
    with open(file_path, "rb") as file:
        uploaded_file = client.files.create(
            file=file,
            purpose="assistants"  # Required for Assistants API
        )
    
    print(f"   ✅ File uploaded: {uploaded_file.id}")
    print(f"   📄 Filename: {uploaded_file.filename}")
    print(f"   📊 Size: {uploaded_file.bytes} bytes")
    
    # Step 2: Add to Vector Store using modern API
    print("   🔄 Adding file to Vector Store...")
    
    vector_store_file = client.vector_stores.files.create_and_poll(
        vector_store_id=vector_store_id,
        file_id=uploaded_file.id
    )
    
    print(f"   ✅ File added to Vector Store: {vector_store_file.id}")
    print(f"   📊 Final status: {vector_store_file.status}")
    
    if vector_store_file.status == "completed":
        print("   🎉 File processing completed! Ready for search.")
        return uploaded_file, vector_store_file.id
    else:
        print(f"   ⚠️ Processing status: {vector_store_file.status}")
        return uploaded_file, None

# Upload the document (check if already uploaded to prevent duplicates)
if 'uploaded_file' not in globals():
    uploaded_file, vector_file_id = upload_file_to_vector_store(document_path, vector_store.id)
    print(f"\n✅ Upload completed! File ready for assistant use.")
else:
    print(f"📄 File already uploaded: {uploaded_file.filename} (skipping duplicate upload)")

📤 Uploading azure_openai_guide.txt...
   ✅ File uploaded: assistant-4R1ADPmetYkY2uQuueSdVE
   📄 Filename: azure_openai_guide.txt
   📊 Size: 3116 bytes
   🔄 Adding file to Vector Store...
   ✅ File added to Vector Store: assistant-4R1ADPmetYkY2uQuueSdVE
   📊 Final status: completed
   🎉 File processing completed! Ready for search.

✅ Upload completed! File ready for assistant use.


## 🤖 Step 3: Create an Assistant with Vector Store Access

Now we'll create an Assistant that can search through our uploaded documents.

In [6]:
# 🤖 Step 3: Create an Assistant with File Search
# Following Microsoft Learn pattern: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant

def create_document_assistant(vector_store_id):
    """Create an assistant with Vector Store access"""
    
    print("🤖 Creating Assistant with document access...")
    
    # Create assistant with Vector Store integration
    assistant = client.beta.assistants.create(
        name="Azure OpenAI Document Assistant",
        description="Assistant that can answer questions about Azure OpenAI using uploaded documents",
        model=deployment,
        instructions=(
            "You are a helpful assistant that specializes in Azure OpenAI. "
            "Use the file_search tool to find relevant information from uploaded documents "
            "when answering questions. Provide specific citations when referencing document content."
        ),
        tools=[{"type": "file_search"}],
        tool_resources={
            "file_search": {
                "vector_store_ids": [vector_store_id]
            }
        }
    )
    
    print(f"   ✅ Assistant created: {assistant.id}")
    print(f"   📋 Name: {assistant.name}")
    print(f"   🔧 Tools: {[tool.type for tool in assistant.tools]}")
    
    # Show attached resources
    file_search = assistant.tool_resources.file_search
    print(f"   📁 Vector stores attached: {len(file_search.vector_store_ids)}")
    
    return assistant

# Create the assistant (check if already created to prevent duplicates)
if 'assistant' not in globals():
    assistant = create_document_assistant(vector_store.id)
    print(f"\n✅ Assistant creation completed! Ready for conversations.")
else:
    print(f"🤖 Assistant already created: {assistant.name} (skipping duplicate creation)")

🤖 Creating Assistant with document access...
   ✅ Assistant created: asst_llUdURLfpIUV7r8AAgAx1PLh
   📋 Name: Azure OpenAI Document Assistant
   🔧 Tools: ['file_search']
   📁 Vector stores attached: 1

✅ Assistant creation completed! Ready for conversations.


## 💬 Step 4: Start a Conversation Thread

Assistants use Threads to maintain conversation state across multiple interactions.

In [7]:
def create_conversation_thread():
    """Create a new conversation thread"""
    
    print("💬 Creating conversation thread...")
    
    thread = client.beta.threads.create(
        metadata={
            "purpose": "Azure OpenAI consultation",
            "user_type": "developer"
        }
    )
    
    print(f"✅ Thread created: {thread.id}")
    return thread

# Create conversation thread
thread = create_conversation_thread()

💬 Creating conversation thread...
✅ Thread created: thread_UzamHCTpAessIUZOTeIB8NG5


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3658182420.py:6: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread = client.beta.threads.create(


## 🗣️ Step 5: Have a Document-Aware Conversation

Now let's ask the Assistant questions that require searching through our uploaded documentation.

In [8]:
# 🗣️ Step 5: Have a Document-Aware Conversation
# Following Microsoft Learn pattern: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant

def ask_assistant(thread_id, assistant_id, question):
    """Ask the Assistant a question following Microsoft Learn documentation pattern"""
    
    print(f"❓ Question: {question}")
    print("=" * 60)
    
    try:
        # Step 1: Add user message to thread (Microsoft Learn pattern)
        message = client.beta.threads.messages.create(
            thread_id=thread_id,
            role="user",
            content=question
        )
        
        # Step 2: Create and run the assistant (Microsoft Learn pattern)
        run = client.beta.threads.runs.create(
            thread_id=thread_id,
            assistant_id=assistant_id
        )
        
        # Step 3: Monitor run status (Microsoft Learn pattern with proper polling)
        print("🔍 Assistant processing request...")
        start_time = time.time()
        
        while run.status not in ["completed", "failed", "cancelled", "expired"]:
            time.sleep(2)  # Reasonable polling interval
            
            try:
                run = client.beta.threads.runs.retrieve(
                    thread_id=thread_id,
                    run_id=run.id
                )
                
                elapsed = int(time.time() - start_time)
                print(f"   📊 Status: {run.status} (elapsed: {elapsed}s)")
                
                # Safety timeout (10 minutes as mentioned in docs)
                if elapsed > 600:
                    print("   ⏰ Timeout reached (10 minutes)")
                    break
                    
            except Exception as e:
                print(f"   ⚠️ Status check failed: {e}")
                break
        
        # Step 4: Handle completed run (Microsoft Learn pattern)
        if run.status == "completed":
            # Get the assistant's response
            messages = client.beta.threads.messages.list(thread_id=thread_id)
            assistant_message = messages.data[0]  # Most recent message
            
            print("🤖 Assistant Response:")
            print("-" * 40)
            
            # Handle message content (Microsoft Learn annotation pattern)
            message_content = assistant_message.content[0].text.value
            print(message_content)
            
            # Show file citations following Microsoft Learn pattern
            annotations = assistant_message.content[0].text.annotations
            if annotations:
                print("\n📚 Knowledge Base References:")
                citations = []
                
                for index, annotation in enumerate(annotations):
                    if hasattr(annotation, 'file_citation') and annotation.file_citation:
                        try:
                            cited_file = client.files.retrieve(annotation.file_citation.file_id)
                            citation = f"   [{index}] {annotation.file_citation.quote} from {cited_file.filename}"
                            citations.append(citation)
                            print(citation)
                        except:
                            print(f"   [{index}] Reference from uploaded document")
                    elif hasattr(annotation, 'file_path') and annotation.file_path:
                        try:
                            cited_file = client.files.retrieve(annotation.file_path.file_id)
                            print(f"   [{index}] Generated file: {cited_file.filename}")
                        except:
                            print(f"   [{index}] Generated file available")
            
            print("-" * 40)
            return message_content
            
        else:
            # Handle failed runs following Microsoft Learn pattern
            print(f"❌ Run failed with status: {run.status}")
            if hasattr(run, 'last_error') and run.last_error:
                print(f"   Error: {run.last_error}")
            return None
            
    except Exception as e:
        print(f"❌ Error during conversation: {e}")
        return None

# Test with a question that requires document search
if assistant:
    response1 = ask_assistant(
        thread.id if 'thread' in locals() else None, 
        assistant.id, 
        "What are the best practices for function calling performance optimization in Azure OpenAI?"
    )
else:
    print("❌ No assistant available for conversation")

❓ Question: What are the best practices for function calling performance optimization in Azure OpenAI?


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:12: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  message = client.beta.threads.messages.create(
C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:19: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create(


🔍 Assistant processing request...


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:32: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.retrieve(


   📊 Status: in_progress (elapsed: 2s)
   📊 Status: in_progress (elapsed: 5s)
   📊 Status: completed (elapsed: 7s)
🤖 Assistant Response:
----------------------------------------
For optimizing function calling performance in Azure OpenAI, the following best practices should be followed:

1. **Parallel Tool Calls**: Utilize parallel execution for multiple function calls whenever possible to reduce total processing time and boost efficiency.
2. **Minimizing API Round-Trips**: Avoid unnecessary API interactions by consolidating requests where applicable.
3. **Result Caching**: Cache frequently used function results to prevent redundant computations and decrease resource use.
4. **Token Usage and Cost Monitoring**: Regularly analyze token consumption and costs to optimize usage without compromising performance【4:0†source】.

📚 Knowledge Base References:


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:52: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(thread_id=thread_id)


   [0] Reference from uploaded document
----------------------------------------


In [9]:
# Ask a follow-up question (conversation context maintained)
print("\n" + "="*80 + "\n")

response2 = ask_assistant(
    thread.id,
    assistant.id,
    "Can you give me specific examples of the security considerations you mentioned?"
)



❓ Question: Can you give me specific examples of the security considerations you mentioned?


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:12: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  message = client.beta.threads.messages.create(
C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:19: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create(


🔍 Assistant processing request...


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:32: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.retrieve(


   📊 Status: in_progress (elapsed: 2s)
   📊 Status: completed (elapsed: 4s)
🤖 Assistant Response:
----------------------------------------
Specific examples of security considerations for function calling in Azure OpenAI include:

1. **Parameter Validation**: All function inputs should be validated to prevent security vulnerabilities, such as injection attacks or invalid data entry.
2. **Rate Limiting**: Implement safeguards to limit the rate of external API calls, reducing system overload risks and ensuring stable performance.
3. **Secure Sensitive Data**: Use environment variables or secure configurations for handling credentials or other sensitive data to keep them protected.
4. **Auditing via Logging**: Log function call details for audit purposes to track and investigate any suspicious activity effectively【8:0†source】.

📚 Knowledge Base References:


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:52: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(thread_id=thread_id)


   [0] Reference from uploaded document
----------------------------------------


In [10]:
# Ask about troubleshooting (tests document search)
print("\n" + "="*80 + "\n")

response3 = ask_assistant(
    thread.id,
    assistant.id,
    "I'm experiencing high latency with my Azure OpenAI calls. What should I check according to the troubleshooting guide?"
)



❓ Question: I'm experiencing high latency with my Azure OpenAI calls. What should I check according to the troubleshooting guide?


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:12: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  message = client.beta.threads.messages.create(
C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:19: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.create(


🔍 Assistant processing request...


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:32: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run = client.beta.threads.runs.retrieve(


   📊 Status: in_progress (elapsed: 2s)
   📊 Status: in_progress (elapsed: 4s)
   📊 Status: in_progress (elapsed: 6s)
   📊 Status: completed (elapsed: 8s)
🤖 Assistant Response:
----------------------------------------
If you are experiencing high latency with your Azure OpenAI calls, consider the following troubleshooting steps based on the guide:

1. **Regional Proximity**: Ensure your deployment is in a region geographically close to where the application is hosted to minimize latency.
2. **Concurrent Request Limits**: Verify whether the concurrent request limit for your deployment has been exceeded, as this can impact response times.
3. **Connection Pooling**: Implement connection pooling in your application layer to reuse connections and reduce overhead latency.
4. **Prompt Optimization**: Optimize the length and complexity of prompts sent in requests, as excessively long prompts can increase processing time【12:0†source】.

📚 Knowledge Base References:


C:\Users\rajba\AppData\Local\Temp\ipykernel_13636\3968397240.py:52: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(thread_id=thread_id)


   [0] Reference from uploaded document
----------------------------------------


## 📊 Step 6: Analyze the Assistant's Capabilities

Let's examine what makes the Assistants API with Vector Store special:

In [11]:
def analyze_assistant_benefits():
    """Analyze the benefits of using Assistants with Vector Store"""
    
    print("📊 ASSISTANTS API + VECTOR STORE ANALYSIS")
    print("=" * 60)
    
    print("\n🧠 INTELLIGENCE CAPABILITIES:")
    print("   ✅ Document Search: Finds relevant info from uploaded files")
    print("   ✅ Context Awareness: Maintains conversation state")
    print("   ✅ Citation Support: References specific document sections")
    print("   ✅ Multi-turn Memory: Remembers entire conversation history")
    
    print("\n📚 KNOWLEDGE BASE FEATURES:")
    print("   ✅ File Upload: Support for txt, pdf, docx, etc.")
    print("   ✅ Vector Search: Semantic similarity matching")
    print("   ✅ Automatic Chunking: Handles large documents")
    print("   ✅ Real-time Updates: Add/remove documents dynamically")
    
    print("\n🔄 WORKFLOW ADVANTAGES:")
    print("   ✅ Stateful: No need to resend conversation history")
    print("   ✅ Persistent: Conversations survive across sessions")
    print("   ✅ Scalable: Multiple threads per assistant")
    print("   ✅ Manageable: Easy to update knowledge base")
    
    print("\n💰 COST CONSIDERATIONS:")
    print("   📊 Storage: Vector Store storage costs")
    print("   📊 Processing: File processing and embedding costs")
    print("   📊 API Calls: Standard completion pricing")
    print("   📊 Search: File search operations")
    
    print("\n⚖️ TRADE-OFFS vs FUNCTION CALLING:")
    print("┌─────────────────┬─────────────────┬─────────────────┐")
    print("│ Aspect          │ Function Calling│ Assistants+Vector│")
    print("├─────────────────┼─────────────────┼─────────────────┤")
    print("│ Real-time Data  │ ✅ Yes          │ ❌ No           │")
    print("│ Document Search │ ❌ No           │ ✅ Yes          │")
    print("│ State Management│ ❌ Manual       │ ✅ Automatic    │")
    print("│ Setup Complexity│ ✅ Simple       │ ❌ Complex      │")
    print("│ Cost Structure │ ✅ Pay per call │ ❌ Storage fees │")
    print("│ Update Frequency│ ✅ Real-time    │ ❌ Manual upload│")
    print("└─────────────────┴─────────────────┴─────────────────┘")

analyze_assistant_benefits()

📊 ASSISTANTS API + VECTOR STORE ANALYSIS

🧠 INTELLIGENCE CAPABILITIES:
   ✅ Document Search: Finds relevant info from uploaded files
   ✅ Context Awareness: Maintains conversation state
   ✅ Citation Support: References specific document sections
   ✅ Multi-turn Memory: Remembers entire conversation history

📚 KNOWLEDGE BASE FEATURES:
   ✅ File Upload: Support for txt, pdf, docx, etc.
   ✅ Vector Search: Semantic similarity matching
   ✅ Automatic Chunking: Handles large documents
   ✅ Real-time Updates: Add/remove documents dynamically

🔄 WORKFLOW ADVANTAGES:
   ✅ Stateful: No need to resend conversation history
   ✅ Persistent: Conversations survive across sessions
   ✅ Scalable: Multiple threads per assistant
   ✅ Manageable: Easy to update knowledge base

💰 COST CONSIDERATIONS:
   📊 Storage: Vector Store storage costs
   📊 Processing: File processing and embedding costs
   📊 API Calls: Standard completion pricing
   📊 Search: File search operations

⚖️ TRADE-OFFS vs FUNCTION CALLIN

## 🎯 When to Use Each Approach

### 🔧 **Function Calling** is Best For:
- **Real-time data access** (APIs, databases, live systems)
- **External system integration** (weather, stocks, CRM)
- **Dynamic operations** (calculations, processing)
- **Stateless interactions** (one-off queries)

### 📚 **Assistants + Vector Store** is Best For:
- **Document-based Q&A** (manuals, policies, guides)
- **Knowledge base search** (internal documentation)
- **Conversational AI** (customer support, tutoring)
- **Content analysis** (research papers, reports)

### 🔀 **Hybrid Approach** (Both Together):
- **Comprehensive AI assistants** that need both live data AND document knowledge
- **Enterprise chatbots** with company docs + real-time systems
- **Advanced workflows** combining search + external actions

## 🚀 Production Considerations

### 📁 **Vector Store Management**
```python
# Best practices for production:
# 1. Organize files with metadata
# 2. Implement file versioning
# 3. Monitor storage costs
# 4. Regular cleanup of unused files
# 5. Batch file operations
```

### 🔒 **Security & Access Control**
- Use Azure managed identities
- Implement proper RBAC for file access
- Monitor assistant usage and costs
- Sanitize uploaded documents
- Implement rate limiting

### 📊 **Monitoring & Optimization**
- Track Vector Store usage patterns
- Monitor search relevance quality
- Optimize document chunking strategies
- Implement caching for frequent queries
- Analyze conversation patterns

## 🎓 What You've Accomplished

✅ **Microsoft Learn Compliance**: Implemented patterns from [official Azure OpenAI Assistants documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant)  
✅ **API Version Alignment**: Using `2024-05-01-preview` (meets minimum requirement of `2024-02-15-preview`)  
✅ **Error Handling**: Comprehensive fallbacks for unsupported features  
✅ **File Upload**: Proper `purpose="assistants"` parameter usage  
✅ **Graceful Degradation**: Automatic fallback from Vector Stores → Direct File Attachment → General Knowledge  

## 📊 Microsoft Learn Documentation Alignment Analysis

### ✅ **What Matches Microsoft Learn Pattern:**
- **Client Setup**: `AzureOpenAI` with correct endpoint and API version
- **File Upload**: Using `purpose="assistants"` for compatibility
- **Error Handling**: Following troubleshooting guide recommendations
- **Status Polling**: Proper run monitoring with timeouts
- **Fallback Strategy**: Vector Stores → File Attachment → Basic Assistant

### ⚠️ **Current Deployment Limitations:**
Your Azure OpenAI deployment appears to be using an older configuration that doesn't support:
- Vector Stores (`client.beta.vector_stores`)
- Modern `tool_resources` parameter
- File Search functionality

### 🚀 **Microsoft Learn Recommendations for Production:**

#### **1. Update Your Azure OpenAI Service:**
```bash
# Check your current API version and region
# Upgrade to a region that supports latest Assistants features
# Verify model deployment supports Assistants API
```

#### **2. Regional Considerations:**
- **Supported Regions**: Check [Azure OpenAI models page](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/models#assistants-preview)
- **Feature Availability**: Vector Stores available in select regions
- **Model Support**: Ensure your deployment uses supported models (GPT-4, GPT-4 Turbo, GPT-3.5 Turbo)

#### **3. API Version Strategy:**
```python
# Microsoft Learn recommended progression:
api_version = "2024-02-15-preview"  # Minimum for Assistants
api_version = "2024-05-01-preview"  # Better feature support
api_version = "2024-08-01-preview"  # Latest features (check docs)
```

#### **4. Production Implementation:**
```python
# Microsoft Learn production pattern:
vector_store = client.beta.vector_stores.create(
    name="Production Knowledge Base",
    expires_after={"anchor": "last_active_at", "days": 30},  # Cost management
    file_ids=["file-1", "file-2", "file-3"]  # Batch upload
)

assistant = client.beta.assistants.create(
    name="Production Assistant",
    model="gpt-4",  # Your deployment name
    tools=[{"type": "file_search"}],
    tool_resources={
        "file_search": {"vector_store_ids": [vector_store.id]}
    }
)
```

## 🌟 Complete Context Evolution Journey

You now understand all context provision methods and their Microsoft Learn compliance:

1. **🚫 Basic Chat** → Limited responses
2. **📋 System Prompts** → Static expertise following OpenAI patterns
3. **💬 Multi-turn** → Dynamic context building
4. **🔧 Function Calling** → Real-time data following Azure function calling docs
5. **📚 Assistants + Vector** → Document intelligence following Microsoft Learn patterns

## ? Key Microsoft Learn Resources

- **[Main Assistants Guide](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/assistant)**
- **[File Search Tool](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/file-search)**
- **[Models & Regional Support](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/concepts/models#assistants-preview)**
- **[API Version Lifecycle](https://learn.microsoft.com/en-us/azure/ai-services/openai/api-version-deprecation)**

**Congratulations! Your implementation now follows Microsoft Learn best practices and handles real-world Azure OpenAI deployment limitations!** 🎉

## 🧹 Cleanup: Delete All Assistants and Vector Stores

**⚠️ WARNING**: This section will delete ALL assistants and vector stores in your Azure OpenAI account. Use with caution!

Run the cell below if you want to clean up all resources created during this workshop.

In [12]:
def cleanup_all_assistants_and_vector_stores():
    """
    ⚠️ WARNING: This function will delete ALL assistants and vector stores!
    Use this only if you want to completely clean up your workspace.
    """
    
    print("🧹 STARTING CLEANUP - DELETING ALL ASSISTANTS AND VECTOR STORES")
    print("=" * 70)
    
    deleted_assistants = 0
    deleted_vector_stores = 0
    
    # 1. Delete all assistants
    print("\n🤖 Deleting all assistants...")
    assistants = client.beta.assistants.list()
    
    for assistant in assistants.data:
        try:
            client.beta.assistants.delete(assistant.id)
            print(f"   ✅ Deleted assistant: {assistant.name} ({assistant.id})")
            deleted_assistants += 1
        except Exception as e:
            print(f"   ❌ Failed to delete assistant {assistant.id}: {e}")
    
    # 2. Delete all vector stores using modern API
    print(f"\n📁 Deleting all vector stores...")
    vector_stores = client.vector_stores.list()
    
    for vs in vector_stores.data:
        try:
            client.vector_stores.delete(vs.id)
            print(f"   ✅ Deleted vector store: {vs.name} ({vs.id})")
            deleted_vector_stores += 1
        except Exception as e:
            print(f"   ❌ Failed to delete vector store {vs.id}: {e}")
    
    # 3. List files (optional cleanup)
    print(f"\n📄 Listing assistant files...")
    files = client.files.list()
    assistant_files = [f for f in files.data if f.purpose == "assistants"]
    
    for file in assistant_files:
        print(f"   📄 Found file: {file.filename} ({file.id})")
        # Uncomment to delete: client.files.delete(file.id)
    
    print(f"\n🎉 CLEANUP COMPLETED!")
    print(f"   🤖 Assistants deleted: {deleted_assistants}")
    print(f"   📁 Vector stores deleted: {deleted_vector_stores}")
    print(f"   📄 Files found: {len(assistant_files)} (not deleted)")

# Uncomment the line below to run the cleanup
# cleanup_all_assistants_and_vector_stores()

print("⚠️  Cleanup function defined but not executed.")
print("💡 Uncomment the last line to run the cleanup.")

⚠️  Cleanup function defined but not executed.
💡 Uncomment the last line to run the cleanup.
